In [ ]:
sharedinputpath = snakemake.params["sharedinputpath"]
desired_regions = snakemake.params.aggregated_regions
weatherdata = snakemake.input.weatherdata

cf_file = snakemake.output.cf_file

In [ ]:
onshore_turbine = snakemake.params.windturbines.get('onshore')
offshore_bottom_turbine = snakemake.params.windturbines.get('offshore_bottom')
offshore_floating_turbine = snakemake.params.windturbines.get('offshore_float')

panel = "CSi"
orientation = "latitude_optimal"

In [ ]:
geodata_files = {
    "onshore": snakemake.input.euroshape,
    "offshore": snakemake.input.eurooffshoreshape,
}

In [ ]:
import logging

import atlite

logging.basicConfig(level=logging.INFO)

import geopandas as gpd
import pandas as pd
import xarray as xr
import numpy as np

In [ ]:
%psource atlite.Cutout.convert_and_aggregate

In [ ]:
boundaries = []
for geodata_file_name, geodata_file_path in geodata_files.items():
    print(geodata_file_path)
    boundaries.append(gpd.read_file(geodata_file_path))

boundaries = pd.concat(boundaries).bounds

In [ ]:
boundaries = boundaries.groupby(lambda x: "bountry").agg(
    {"minx": "min", "miny": "min", "maxx": "max", "maxy": "max"}
)

boundaries["minx"] = boundaries["minx"] - 2
boundaries["miny"] = boundaries["miny"] - 2
boundaries["maxx"] = boundaries["maxx"] + 2
boundaries["maxy"] = boundaries["maxy"] + 2

In [ ]:
ds = xr.open_dataset(weatherdata, chunks="auto")

In [ ]:
cutout = atlite.Cutout(
    path="../3_intermediate_data/intermediatecutout.nc",
    data=ds.sel(
        x=slice(
            boundaries.loc["bountry", "minx"],
            boundaries.loc["bountry", "maxx"],
        ),
        y=slice(
            boundaries.loc["bountry", "miny"],
            boundaries.loc["bountry", "maxy"],
        ),
    ),
)

cutout.prepare()

In [ ]:
# Wind bias-correction - 100m wind speed

In [ ]:
if snakemake.params.bias_correction:
    # Path to bias correction ratios
    windbiaspath = snakemake.input.biaswinddata

    # Get wind speed at 100m from cutout
    wnd_100m = cutout.data["wnd100m"]
    attrs = wnd_100m.attrs

    # Load dataset containing bias correction ratios
    gwa_ratio = (xr.open_dataarray(windbiaspath, chunks="auto")
                 .rename({"longitude": "x", "latitude": "y"}))

    # Bias-correction based on comparing ERA5 and GWA4 data between 2008-2017. GWA4
    # is regridded to ERA5 resolution for comparison and output is a scaling
    # factor about 1, i.e. above 1 increases wind speeds.

    gwa_ratio=gwa_ratio.interp_like(wnd_100m)

    cutout.data["wnd100m"] = (
         wnd_100m * gwa_ratio
     ).assign_attrs(attrs)  # copy also the attributes

In [ ]:
out=[]
techs=[]

In [ ]:
# Solar CF

In [ ]:
%psource atlite.Cutout.convert_and_aggregate

In [ ]:
cf_solar = cutout.pv(
    panel=panel,
    orientation=orientation,
    capacity_factor_timeseries=True,
).astype(np.float32)*snakemake.params.solarpv_af

out.append(cf_solar)
techs.append("Solar")

In [ ]:
# Onshore wind CF

In [ ]:
cf_windon = cutout.wind(
    turbine=onshore_turbine, 
    capacity_factor_timeseries=True,
    smooth=snakemake.params.wind_smooth
).astype(np.float32)*snakemake.params.windon_af

out.append(cf_windon)
techs.append("Windonshore")

In [ ]:
# Offshore wind CF

In [ ]:
cf_windoff_bottom = cutout.wind(
    turbine=offshore_bottom_turbine, 
    capacity_factor_timeseries=True,
    smooth=snakemake.params.wind_smooth
).astype(np.float32)*snakemake.params.windoff_bottom_af

out.append(cf_windoff_bottom)
techs.append("Windoffshore")

In [ ]:
# Offshore floating CF

In [ ]:
if snakemake.params.windoff_floating:
    cf_windoff_floating = cutout.wind(
        turbine=offshore_floating_turbine, 
        capacity_factor_timeseries=True,
        smooth=snakemake.params.wind_smooth
    ).astype(np.float32)*snakemake.params.windoff_floating_af

    out.append(cf_windoff_floating)
    techs.append("WindoffshoreFloat")

In [ ]:
cf = xr.concat(
    out,
    pd.Index(techs, name="Tech"),
)
# rename variable name from "specific generation" to "capacity factor"
cf = cf.rename("capacity factor")

In [ ]:
cf.to_netcdf(
    cf_file,
    encoding={
        "capacity factor": {
            "dtype": "int16",
            "scale_factor": 0.001,
            "_FillValue": -99,
            "zlib": True,
            "complevel": 1,
        }
    },
)